In [1]:
print("5")

5


In [2]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
train_df = pd.read_csv("train_new.csv")
val_df = pd.read_csv("validation_new.csv")
test_df = pd.read_csv("test_new.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (67533, 37)
Validation: (14471, 37)
Test: (14472, 37)


In [5]:
train_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'total_items', 'total_price', 'total_freight_value',
       'avg_item_price', 'total_payment_value', 'max_installments',
       'payment_count', 'main_payment_type', 'unique_products',
       'unique_sellers', 'purchase_month', 'purchase_day', 'purchase_weekday',
       'purchase_hour', 'is_weekend', 'multi_seller', 'freight_ratio',
       'freight_per_item', 'total_weight', 'total_volume', 'avg_volume',
       'max_volume', 'distance_km', 'is_cross_state', 'is_late'],
      dtype='object')

In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for df in [train_df, val_df, test_df]:
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

In [ ]:
def create_date_features(df):

    df = df.copy()

    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_day"] = df["order_purchase_timestamp"].dt.day
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.weekday
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    return df

In [8]:
train_df = create_date_features(train_df)
val_df = create_date_features(val_df)
test_df = create_date_features(test_df)

In [9]:
target = "is_late"

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

In [10]:
X_train = train_df.drop(columns=[target])
X_val = val_df.drop(columns=[target])
X_test = test_df.drop(columns=[target])

In [11]:
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

In [12]:
future_columns = [
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

In [13]:
raw_date_columns = [
    "order_purchase_timestamp"
]

In [14]:
drop_columns = id_columns + future_columns + raw_date_columns

X_train = X_train.drop(columns=drop_columns, errors="ignore")
X_val = X_val.drop(columns=drop_columns, errors="ignore")
X_test = X_test.drop(columns=drop_columns, errors="ignore")

In [15]:
categorical_features = [
    "customer_city",
    "customer_state",
    "main_payment_type"
]

categorical_features = [
    col for col in categorical_features
    if col in X_train.columns
]

In [16]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

In [17]:
print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['customer_zip_code_prefix', 'total_items', 'total_price', 'total_freight_value', 'avg_item_price', 'total_payment_value', 'max_installments', 'payment_count', 'unique_products', 'unique_sellers', 'is_weekend', 'multi_seller', 'freight_ratio', 'freight_per_item', 'total_weight', 'total_volume', 'avg_volume', 'max_volume', 'distance_km', 'is_cross_state']

Categorical features:
['customer_city', 'customer_state', 'main_payment_type']


In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [19]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

In [20]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [21]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Train transformed shape:", X_train_processed.shape)
print("Validation transformed shape:", X_val_processed.shape)
print("Test transformed shape:", X_test_processed.shape)


Train transformed shape: (67533, 3701)
Validation transformed shape: (14471, 3701)
Test transformed shape: (14472, 3701)


In [23]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))
print(feature_names)

Number of final features: 3701
['num__customer_zip_code_prefix' 'num__total_items' 'num__total_price' ...
 'cat__main_payment_type_credit_card' 'cat__main_payment_type_debit_card'
 'cat__main_payment_type_voucher']


In [24]:
feature_names = preprocessor.get_feature_names_out()

print("Number of output features:", len(feature_names))
print("Transformed columns:", X_train_processed.shape[1])

Number of output features: 3701
Transformed columns: 3701


In [26]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("\nTransformed:")
print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)
print("X_test_processed:", X_test_processed.shape)

print("\nFeature names:", len(feature_names))

X_train shape: (67533, 31)
X_val shape: (14471, 31)
X_test shape: (14472, 31)

Transformed:
X_train_processed: (67533, 3701)
X_val_processed: (14471, 3701)
X_test_processed: (14472, 3701)

Feature names: 3701


In [29]:
os.makedirs("features", exist_ok=True)

In [30]:
preprocessor.fit_transform(X_train)

<67533x3701 sparse matrix of type '<class 'numpy.float64'>'
	with 1553259 stored elements in Compressed Sparse Row format>

In [36]:
joblib.dump( preprocessor, "features/preprocessor.joblib" ) 
print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [37]:
joblib.dump( feature_names, "features/feature_names.joblib" ) 
print(f"Saved {len(feature_names)} feature names.")

Saved 3701 feature names.


In [33]:
feature_metadata = { "numerical_features": numerical_features, "categorical_features": categorical_features, "final_feature_names": feature_names.tolist() } 
joblib.dump( feature_metadata, "features/feature_metadata.joblib" ) 
print("Feature metadata saved successfully.")

Feature metadata saved successfully.


In [38]:
# Save processed feature matrices 
joblib.dump( X_train_processed, "features/X_train_processed.joblib" ) 
joblib.dump( X_val_processed, "features/X_val_processed.joblib" ) 
joblib.dump( X_test_processed, "features/X_test_processed.joblib" ) 
print("Processed feature matrices saved successfully.")

Processed feature matrices saved successfully.


In [39]:
joblib.dump( y_train, "features/y_train.joblib" ) 
joblib.dump( y_val, "features/y_val.joblib" ) 
joblib.dump( y_test, "features/y_test.joblib" ) 
print("Target variables saved successfully.")

Target variables saved successfully.


In [40]:
print("Artifacts:") 
for file in os.listdir("features"): print(" -", file)

Artifacts:
 - best_model.joblib
 - confusion_matrix.png
 - feature_metadata.joblib
 - feature_names.joblib
 - final_test_results.csv
 - model_summary.csv
 - precision_recall_curve.png
 - preprocessor.joblib
 - roc_curve.png
 - test_classification_report.csv
 - test_confusion_matrix.csv
 - validation_results.csv
 - X_test_processed.joblib
 - X_train_processed.joblib
 - X_val_processed.joblib
 - y_test.joblib
 - y_train.joblib
 - y_val.joblib


In [41]:
print("Training target distribution:") 
print(y_train.value_counts()) 
print("\nTraining target percentage:") 
print(y_train.value_counts(normalize=True) * 100) 
print("\nValidation target distribution:") 
print(y_val.value_counts()) 
print("\nValidation target percentage:") 
print(y_val.value_counts(normalize=True) * 100)

Training target distribution:
is_late
0    62054
1     5479
Name: count, dtype: int64

Training target percentage:
is_late
0    91.886929
1     8.113071
Name: proportion, dtype: float64

Validation target distribution:
is_late
0    13297
1     1174
Name: count, dtype: int64

Validation target percentage:
is_late
0    91.887223
1     8.112777
Name: proportion, dtype: float64


In [42]:
def create_freight_features(df):
    df = df.copy()

    df["freight_to_price_ratio"] = (
        df["total_freight_value"] /
        df["total_price"].replace(0, np.nan)
    )

    df["freight_per_item"] = (
        df["total_freight_value"] /
        df["total_items"].replace(0, np.nan)
    )

    return df

In [43]:
train_df = create_freight_features(train_df)
val_df = create_freight_features(val_df)
test_df = create_freight_features(test_df)

In [44]:
def create_order_features(df):
    df = df.copy()

    df["items_per_seller"] = (
        df["total_items"] /
        df["unique_sellers"].replace(0, np.nan)
    )

    df["is_multi_seller"] = (
        df["unique_sellers"] > 1
    ).astype(int)

    return df

In [45]:
def create_order_features(df):
    df = df.copy()

    # Items per seller
    df["items_per_seller"] = (
        df["total_items"] /
        df["unique_sellers"].replace(0, np.nan)
    )

    # More than one seller in the order
    df["is_multi_seller"] = (
        df["unique_sellers"] > 1
    ).astype(int)

    # Weekend purchase
    df["is_weekend"] = (
        df["purchase_weekday"] >= 5
    ).astype(int)

    return df

In [46]:
train_df = create_order_features(train_df)
val_df = create_order_features(val_df)
test_df = create_order_features(test_df)

In [47]:
train_df[
    [
        "items_per_seller",
        "is_multi_seller",
        "is_weekend"
    ]
].head()

,items_per_seller,is_multi_seller,is_weekend
0,1.0,0,0
1,1.0,0,0
2,1.0,0,0
3,1.0,0,1
4,1.0,0,0


In [48]:
def create_geographic_features(df):
    df = df.copy()

    if "customer_state" in df.columns and "seller_state" in df.columns:

        df["is_cross_state"] = (
            df["customer_state"] != df["seller_state"]
        ).astype(int)

    return df

In [49]:
train_df = create_geographic_features(train_df)
val_df = create_geographic_features(val_df)
test_df = create_geographic_features(test_df)

In [50]:
for df in [train_df, val_df, test_df]:

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [51]:
target = "is_late"

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

In [52]:
X_train = train_df.drop(columns=[target])
X_val = val_df.drop(columns=[target])
X_test = test_df.drop(columns=[target])

In [53]:
print(train_df.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_items', 'total_price', 'total_freight_value', 'avg_item_price', 'total_payment_value', 'max_installments', 'payment_count', 'main_payment_type', 'unique_products', 'unique_sellers', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour', 'is_weekend', 'multi_seller', 'freight_ratio', 'freight_per_item', 'total_weight', 'total_volume', 'avg_volume', 'max_volume', 'distance_km', 'is_cross_state', 'is_late', 'purchase_year', 'freight_to_price_ratio', 'items_per_seller', 'is_multi_seller']
